In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
Base = "/content/drive/MyDrive/WasteWise/"

### Part 1: Dataset Arranging

In [ ]:
import pandas as pd

demand = pd.read_csv("/content/drive/MyDrive/WasteWise/train.csv")
print(demand.shape)
print(demand.week.min(), demand.week.max())
demand.head()

(456548, 9)
1 145


,id,week,center_id,meal_id,checkout_price,base_price,emailer_for_promotion,homepage_featured,num_orders
0,1379560,1,55,1885,136.83,152.29,0,0,177
1,1466964,1,55,1993,136.83,135.83,0,0,270
2,1346989,1,55,2539,134.86,135.86,0,0,189
3,1338232,1,55,2139,339.50,437.53,0,0,54
4,1448490,1,55,2631,243.50,242.50,0,0,40


In [ ]:
#Simple version to see
simple = demand[["week", "center_id", "meal_id", "num_orders"]]
simple.head()

,week,center_id,meal_id,num_orders
0,1,55,1885,177
1,1,55,1993,270
2,1,55,2539,189
3,1,55,2139,54
4,1,55,2631,40


* id - row indetifier
* week - 1 to 145
* center_id - which kitchen
* meal_id - which meal
* checkout_price - what the customer paid
* base_price - list prie
* emailer_for promotion - if promo email sent out
* homepage_featured - 1 if featured on home page
* num_order - the thing we predict

The project

A meal delivery company has kitchens in lots of cities. Every week each kitchen preps meals. Prep too much and food in the bin. Prep too little and customers don’t get fed.

We predict next week’s demand so they prep the right amount. Our headline isn’t “our model is accurate” so it’s kilograms of food waste avoided.

The data

Three files from Kaggle:

train.csv — One row = one meal, at one kitchen, in one week, with how many were ordered. 456,548 rows.
fulfilment_center_info.csv — facts about each kitchen (city, size)
meal_info.csv — facts about each meal (category, cuisine)

In [ ]:
import pandas as pd

demand = pd.read_csv("/content/drive/MyDrive/WasteWise/train.csv")
centers = pd.read_csv("/content/drive/MyDrive/WasteWise/fulfilment_center_info.csv")
meals   = pd.read_csv("/content/drive/MyDrive/WasteWise/meal_info.csv")

#order rows only contain center_id = 55. They don't know about center 55. Facts live in another file so merging.
df = demand.merge(centers, on="center_id").merge(meals, on="meal_id")
# Now every row has the kitchen's city, region, and size.
# discount depth — a real signal, cheap to add
print(demand.shape)
print(df.shape)
df.head(3)

print(df.columns.tolist())


(456548, 9)
(456548, 15)
['id', 'week', 'center_id', 'meal_id', 'checkout_price', 'base_price', 'emailer_for_promotion', 'homepage_featured', 'num_orders', 'city_code', 'region_code', 'center_type', 'op_area', 'category', 'cuisine']


Discount : base_price is the list price. Checkout_price is what they actually paid. Subtract to get dollars off. Divide by base_price to turn it into a percentage.  Since discounts spike demand, if the model sees the big dicount it won't be suprised by spike in sales.

In [ ]:
# readable names
df = df.rename(columns={
    "center_id":            "kitchen",
    "meal_id":              "meal",
    "num_orders":           "orders",
    "checkout_price":       "price_paid",
    "base_price":           "list_price",
    "emailer_for_promotion":"promo_email",
    "homepage_featured":    "on_homepage",
    "city_code":            "city",
    "region_code":          "region",
    "center_type":          "kitchen_type",
    "op_area":              "kitchen_size",
})

# how far below list price the meal sold (negative = sold above list)
df["price_cut"] = (df["list_price"] - df["price_paid"]) / df["list_price"]

# one meal at one kitchen, weeks in order
df = df.sort_values(["kitchen", "meal", "week"])

# each meal's own past demand ( shifts it so contains information we are trying to predict)
g = df.groupby(["kitchen", "meal"])["orders"]
df["orders_last_week"] = g.shift(1)
df["orders_2wk_ago"]   = g.shift(2)
df["orders_avg_4wk"]   = g.transform(lambda s: s.shift(1).rolling(4).mean())

HISTORY = ["orders_last_week", "orders_2wk_ago", "orders_avg_4wk"]

# split by time, never at random — everyone imports this
def split(d):
    return (d[d.week <= 125], #models training set
            d[(d.week > 125) & (d.week <= 135)],#Validation - tune settings
            d[d.week > 135]) #test, only at the end


Why by time and not randomly - random split would scatter week 140's rows into training and the model would learn from week 140, and then get tested on week 140. It would score great and that would be worthless.

In [ ]:

train, val, test = split(df.dropna(subset=HISTORY))

def mae(actual, predicted):
    return (actual - predicted).abs().mean()

print("baseline 1 — same as last week: ", mae(test.orders, test.orders_last_week))
print("baseline 2 — 4-week average:    ", mae(test.orders, test.orders_avg_4wk))

baseline 1 — same as last week:  79.13263440204803
baseline 2 — 4-week average:     81.7012678288431


The model should beat 79.13.

What this did:
*   Merged the three files so every row carries its kitchen and meal attributes. A model now reads one row t a time and can use the facts of a larger kitches sell more of a certain food.
*   Renamed columns to simpler language
*   Built price_cut = ( ist_price - price_paid ) / list_price. Expressed as a fraction, not dollars.
*   Sorted by kitchen, meal, week so each meal 's history reads from top to bottom in order, needed for the lag features to mean anything
* Built three history features:
  * orders_last_week - demand one week ago
  * orders_2wk_ago - demand two weeks ago
  *orders_avg_4wk - meand of the four weeks before this one
Grouping keeps on'es kitchen from anothers. The window excludes the current week so the value we are predicting isn't contained in the window.
* Split by time
  * Train: weeks 1 - 125
  * Validation: weeks 126 - 135
  * Test: weeks 136-145

Rows missing history ( the earliest weeks per mean ) are droped

Basline results ( week 136-145)
- Same as last week - 79.13 ( MAE orders )
- 4-week average - 81.70 ( MAE orders )

79.13 is the number to beat.


### Part 2 - Waste metrics ( order to kilograms )

Our forecast is by orders. The climate claim needs kilograms. So we need a defensible conversion: how much food is in one prepared meal?
Why we brought in a second dataset

*   We looked for real measured food waste rather than inventing a number. The University Canteen Food Waste dataset records daily waste by meal period, canteen section and food category, with weight in kg and cost.

What it could and couldn’t give us
* It couldn’t give us kg per meal. Each row aggregates one session — one day, one meal period, one section, one food category — across however many people ate.

With no meals_served column, there is no denominator, so the 2.59 kg mean below is kg per session, not per meal. Using it as a per-meal factor would inflate our results by orders of magnitude.
It could give us food costs. Unit_Price_per_kg is measured, and it anchors our portion estimate against a second, independent source.


In [ ]:
#Has kg wastes per section per meal per period, per day. It lacks how many meals were served
canteen = pd.read_csv("/content/drive/MyDrive/WasteWise/Dataset Propely.csv")
print(canteen.columns.tolist())
canteen.head()

['Date', 'Meal', 'Canteen_Section', 'Food_Category', 'Waste_Weight_kg', 'Unit_Price_per_kg', 'Cost_Loss']


,Date,Meal,Canteen_Section,Food_Category,Waste_Weight_kg,Unit_Price_per_kg,Cost_Loss
0,2025-07-30,Dinner,B,Vegetables,1.50,3.0,4.50
1,2025-06-15,Breakfast,B,Rice,3.69,2.0,7.38
2,2025-07-29,Breakfast,A,Soup,1.54,1.5,2.31
3,2025-07-17,Breakfast,A,Soup,1.81,1.5,2.71
4,2025-07-03,Dinner,D,Rice,4.69,2.0,9.38


In [ ]:
print(canteen.groupby("Food_Category")["Waste_Weight_kg"].agg(["mean", "count"]))
print("\noverall mean kg per record:", canteen["Waste_Weight_kg"].mean())

                   mean  count
Food_Category                 
Meat           2.480781    653
Rice           2.609954    659
Soup           2.662826    637
Vegetables     2.592074    651

overall mean kg per record: 2.5859884615384616


THe kg is per person and far to big for a person's plate which tells we cannot use this ingformation. Missing column told us we can use it and the 2.59 confirmed the scale was wrong.

Next we group the canteen data by good type and averages the price per kg. We get what a kilogram of food costs.

In [ ]:
print(canteen.groupby("Food_Category")["Unit_Price_per_kg"].mean())
print("\nmean meal price in main data:", df["price_paid"].mean())

Food_Category
Meat          8.0
Rice          2.0
Soup          1.5
Vegetables    3.0
Name: Unit_Price_per_kg, dtype: float64

mean meal price in main data: 332.2389325547369


Our 0.4 kg assumption - neither dataset contains a per-meal weight so the conversion factor comes from outdie. We used 0.4 kg per prepared meal, taken from USDA FNDDS portion weights. Checked with own data. Check - canteen dataset gives measured good cost per kg, the Genpact datasert gived the average meal price. Together they imply a weight.

we report all waste results at 0.3, 0.4 and 0.5 kg per meal, so the conclusion does not rest on this one number.

Step 1 — convert currency

The meal costs ₹332. At roughly ₹88 to the dollar, that’s about $3.80.

Why rupees: the Genpact dataset is from an Indian company. A $332 meal would be absurd; ₹332 is a normal delivery order.

Step 2 — price to ingredient cost

Restaurants don’t spend the whole price on food. Most goes to staff, rent, delivery, profit. Ingredients are typically around 30% of the menu price.

$3.80 × 0.30 ≈ $1.14 of actual food

Step 3 — cost to weight

Now use the canteen’s measured prices. Vegetables ran $3/kg.

$1.14 ÷ $3/kg ≈ 0.38 kg

Which rounds to 0.4 kg — the same figure USDA gave us from an entirely different direction.

In [ ]:
KG_PER_MEAL = 0.4          # USDA FNDDS portion estimate
FOOD_COST_RATE = 0.30      # ingredients as a share of meal price
# If you prepped 100 and sold 80 - over = 20
def waste_report(actual, prepped, avg_price):
    over  = (prepped - actual).clip(lower=0)   # prepped, not sold
    under = (actual - prepped).clip(lower=0)   # demand unmet

    return {
        "meals_wasted":   round(over.sum()),
        "kg_wasted":      round(over.sum() * KG_PER_MEAL, 1),
        "cost_wasted_rs": round(over.sum() * avg_price * FOOD_COST_RATE),
        "meals_unserved": round(under.sum()),
        "revenue_lost_rs":round(under.sum() * avg_price),
    }

avg_price = test["price_paid"].mean()

print("same as last week:", waste_report(test.orders, test.orders_last_week, avg_price))
print("4-week average:   ", waste_report(test.orders, test.orders_avg_4wk,   avg_price))

same as last week: {'meals_wasted': 1350403, 'kg_wasted': np.float64(540161.2), 'cost_wasted_rs': 138312583, 'meals_unserved': 1246097, 'revenue_lost_rs': 425430764}
4-week average:    {'meals_wasted': 1500708, 'kg_wasted': np.float64(600283.0), 'cost_wasted_rs': 153707249, 'meals_unserved': 1180074, 'revenue_lost_rs': 402889981}


In [ ]:
print("rows in test:", len(test))
print("total orders:", test.orders.sum())
print("waste as % of demand:", 1350403 / test.orders.sum())

rows in test: 32812
total orders: 7459820
waste as % of demand: 0.1810235367609406


In [ ]:
for m in [1.0, 1.05, 1.1, 1.15, 1.2, 1.25]:
    r = waste_report(val.orders, val.orders_last_week * m, val.price_paid.mean())
    total = r["cost_wasted_rs"] + r["revenue_lost_rs"]
    print(m, "total cost:", total)

1.0 total cost: 864093551
1.05 total cost: 844150951
1.1 total cost: 831987141
1.15 total cost: 826856848
1.2 total cost: 827858222
1.25 total cost: 833781081


In [ ]:
for m in [1.10, 1.12, 1.14, 1.15, 1.16, 1.18, 1.20]:
    r = waste_report(val.orders, val.orders_last_week * m, val.price_paid.mean())
    print(m, r["cost_wasted_rs"] + r["revenue_lost_rs"])

1.1 831987141
1.12 829118017
1.14 827343493
1.15 826856848
1.16 826622016
1.18 826792793
1.2 827858222


In [ ]:
OPTIMAL_MULTIPLIER = 1.2

Prepping to the forecast is not optimal

1.   List item
2.   List item



Our cost model is **asymmetric**: a wasted meal loses only ingredients (~30% of price), while an unserved order loses the full sale. Underprepping is therefore roughly **3× more costly** than overprepping.

This means the cost-minimizing prep quantity sits **above** the demand forecast, not at it.

We swept a multiplier on the forecast across validation weeks (126–135) and measured total cost (wasted ingredients + lost revenue):

| multiplier | total cost (₹) |
|:---|---:|
| 1.00 | 864,093,551 |
| 1.10 | 831,987,141 |
| 1.14 | 827,343,493 |
| **1.16** | **826,622,016** |
| 1.20 | 827,858,222 |
| 1.25 | 833,781,081 |

**Result:** prep **16% above forecast**. Saves ₹37.5M (4.3%) versus prepping to forecast exactly.

# Part 3 - Modeling

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
import pandas as pd
import numpy as np

# Scikit-learn builds the machine learning model.
# Pandas handles the dataset.
# NumPy helps with numerical calculations.



In [ ]:
print(df.columns.tolist())

['id', 'week', 'kitchen', 'meal', 'price_paid', 'list_price', 'promo_email', 'on_homepage', 'orders', 'city', 'region', 'kitchen_type', 'kitchen_size', 'category', 'cuisine', 'price_cut', 'orders_last_week', 'orders_2wk_ago', 'orders_avg_4wk']


In [ ]:
# Information about the kitchen and meal
categorical_features = [
    "kitchen",
    "meal",
    "city",
    "region",
    "kitchen_type",
    "category",
    "cuisine"
]

# Numerical information and previous demand
numerical_features = [
    "price_paid",
    "list_price",
    "price_cut",
    "promo_email",
    "on_homepage",
    "kitchen_size",
    "orders_last_week",
    "orders_2wk_ago",
    "orders_avg_4wk"
]

# Combine all features
FEATURES = categorical_features + numerical_features

# What we want the AI to predict
TARGET = "orders"

print("Selected features:", FEATURES)
print("Target:", TARGET)

Selected features: ['kitchen', 'meal', 'city', 'region', 'kitchen_type', 'category', 'cuisine', 'price_paid', 'list_price', 'price_cut', 'promo_email', 'on_homepage', 'kitchen_size', 'orders_last_week', 'orders_2wk_ago', 'orders_avg_4wk']
Target: orders


In [ ]:
from sklearn.compose import ColumnTransformer #to apply dif changes to dif columns
from sklearn.preprocessing import OneHotEncoder#turn categories into numbers

# Create our data preparation tool
preprocessor = ColumnTransformer(
    transformers=[

        # Convert categorical columns into numerical columns
        ("categories",
         # Produce a regular dense table instead of sparse data
         OneHotEncoder(
             handle_unknown="ignore",
             sparse_output=False
         ),
         # Apply this conversion to our categorical features
         categorical_features)
    ],

    # Keep numerical columns unchanged
    remainder="passthrough"
)

# Confirm that preparation is complete
print("Data preprocessing setup complete")

Data preprocessing setup complete


X represents the information given to the model.

y represents the correct answers the model needs to learn.

In [ ]:
# Select the input information
X_train = train[FEATURES] # weeks 1-125
X_val = val[FEATURES] #weeks 126-135
X_test = test[FEATURES] #weeks 136-145

# Select the answers we want the AI to learn
y_train = train[TARGET] #training - teaching AI
y_val = val[TARGET] #validation - helping AI
y_test = test[TARGET] #testing - testing AI, final eval

print("Training rows:", len(X_train)) #shows how many rows being used for training
print("Validation rows:", len(X_val)) # for validation
print("Testing rows:", len(X_test))# for testing

Training rows: 376563
Validation rows: 32799
Testing rows: 32812


Gradient Boosting builds multiple decision trees.
We chose it because food demand relies on multiple factors, such as

*   Location
*   Price
*   History

Gradient boosting learns complicated relationships

In [ ]:
# Import Gradient Boosting - machine learning algorithm we will use
from sklearn.ensemble import HistGradientBoostingRegressor

# connect our data preparation and AI model
from sklearn.pipeline import Pipeline

model = Pipeline([

    # Convert the categorical data into numerical information
    ("preprocessing", preprocessor),

    # use Gradient Boosting to learn patterns and predict meal orders
    ("regressor", HistGradientBoostingRegressor(

        max_iter=200, # number of learning stages / number of rounds it takes to fix mistake
        learning_rate=0.08, # # Controls how much each stage changes the model
        max_leaf_nodes=31,# Controls how complex each decision tree can become
        l2_regularization=1.0, # so model doesn't memorize training data, instead learns patterns

        # Makes the results reproducible
        random_state=42
    ))
])

print("Gradient Boosting model created successfully")

Gradient Boosting model created successfully


In [ ]:
# Start training the Gradient Boosting model using training data
model.fit(X_train, y_train)

# Print a message once training is finished
print("Model training completed successfully")

Model training completed successfully


In [ ]:
# Ask trained AI to predict meal orders for the validation data
val_predictions = model.predict(X_val)

# Calculate the average difference between actual and predicted orders
val_mae = mae(y_val, pd.Series(val_predictions, index=y_val.index))

# Display the model's validation error
print("Validation MAE:", val_mae)

Validation MAE: 78.68178601856673


Baseline 1 - next week's orders will be exactly the same as last week's orders

Baseline 2 - Instead of looking at just last week, it calculates the average orders from the previous four weeks.


In [ ]:
# Calculate the error when predicting using last week's orders
baseline_1_mae = mae(y_val, val["orders_last_week"])

# Calculate the error when predicting using the previous 4-week average
baseline_2_mae = mae(y_val, val["orders_avg_4wk"])

# Display all three model results
print("Baseline 1 (last week):", baseline_1_mae)
print("Baseline 2 (4-week average):", baseline_2_mae)
print("Gradient Boosting:", val_mae)

Baseline 1 (last week): 119.09881398823134
Baseline 2 (4-week average): 120.87178724961127
Gradient Boosting: 78.68178601856673


In [ ]:
# Calculate the percentage improvement over the first baseline
improvement_1 = (
    (baseline_1_mae - val_mae) / baseline_1_mae
) * 100

# Calculate the percentage improvement over the second baseline
improvement_2 = (
    (baseline_2_mae - val_mae) / baseline_2_mae
) * 100

# Display both improvements
print("Improvement over last-week baseline:",
      round(improvement_1, 2), "%")

print("Improvement over 4-week baseline:",
      round(improvement_2, 2), "%")

Improvement over last-week baseline: 33.94 %
Improvement over 4-week baseline: 34.9 %


testing three versions of Gradient Boosting to see which is bets.

In [ ]:
# Import the Gradient Boosting algorithm
from sklearn.ensemble import HistGradientBoostingRegressor

# Import Pipeline to connect preprocessing and the model
from sklearn.pipeline import Pipeline

# Create different model settings to compare
settings = [
    {"max_iter": 100, "learning_rate": 0.1, "max_leaf_nodes": 31},
    {"max_iter": 200, "learning_rate": 0.08, "max_leaf_nodes": 31},
    {"max_iter": 300, "learning_rate": 0.05, "max_leaf_nodes": 20}
]

# Store the results from each experiment
tuning_results = []

# Try each set of settings one by one
for setting in settings:

    # Build a new model using the current settings
    test_model = Pipeline([
        ("preprocessing", preprocessor),

        ("regressor", HistGradientBoostingRegressor(
            # Number of learning stages
            max_iter=setting["max_iter"],

            # How strongly each stage affects the model
            learning_rate=setting["learning_rate"],

            # Maximum number of leaves in each tree
            max_leaf_nodes=setting["max_leaf_nodes"],

            # Keep results reproducible
            random_state=42
        ))
    ])

    # Train the model using training data only
    test_model.fit(X_train, y_train)

    # Predict orders using validation data
    predictions = test_model.predict(X_val)

    # Calculate prediction error
    error = mae(
        y_val,
        pd.Series(predictions, index=y_val.index)
    )

    # Save the settings and their validation error
    tuning_results.append({
        "settings": setting,
        "validation_mae": error
    })

    # Display the result
    print(setting, "MAE:", round(error, 4))

{'max_iter': 100, 'learning_rate': 0.1, 'max_leaf_nodes': 31} MAE: 79.7685
{'max_iter': 200, 'learning_rate': 0.08, 'max_leaf_nodes': 31} MAE: 78.7739
{'max_iter': 300, 'learning_rate': 0.05, 'max_leaf_nodes': 20} MAE: 79.0921


Second one was best

In [ ]:
# Find the experiment with the lowest validation error
best_result = min(
    tuning_results,
    key=lambda result: result["validation_mae"]
)

# Display the best settings
print("Best settings:", best_result["settings"])

# Display the best validation error
print("Best validation MAE:", best_result["validation_mae"])

Best settings: {'max_iter': 200, 'learning_rate': 0.08, 'max_leaf_nodes': 31}
Best validation MAE: 78.77391215709483


In [ ]:
# Get the best settings from our experiment
best_settings = best_result["settings"]

# Build our final model using those settings
best_model = Pipeline([
    # Prepare the data
    ("preprocessing", preprocessor),

    # Create the Gradient Boosting model
    ("regressor", HistGradientBoostingRegressor(
        max_iter=best_settings["max_iter"],
        learning_rate=best_settings["learning_rate"],
        max_leaf_nodes=best_settings["max_leaf_nodes"],
        random_state=42
    ))
])

# Train the selected model on the training data
best_model.fit(X_train, y_train)

# Confirm that training is complete
print("Best model trained successfully")

Best model trained successfully


Testing Model with second version

In [ ]:
# Ask our best model to predict orders for the test dataset
test_predictions = best_model.predict(X_test)

# Calculate the model's average prediction error
test_model_mae = mae(
    y_test,
    pd.Series(test_predictions, index=y_test.index)
)

# Calculate the first baseline's error on the same test data
test_baseline_1 = mae(
    y_test,
    test["orders_last_week"]
)

# Calculate the second baseline's error on the same test data
test_baseline_2 = mae(
    y_test,
    test["orders_avg_4wk"]
)

# Display all three results
print("Gradient Boosting MAE:", round(test_model_mae, 4))
print("Last-week baseline MAE:", round(test_baseline_1, 4))
print("4-week average baseline MAE:", round(test_baseline_2, 4))

Gradient Boosting MAE: 64.5201
Last-week baseline MAE: 79.1326
4-week average baseline MAE: 81.7013


Find out which approach results in the least estimated food waste and financial loss.

In [ ]:
test["predicted"] = test_predictions

avg_price = test["price_paid"].mean()

print("baseline (last week):", waste_report(test.orders, test.orders_last_week, avg_price))
print("model, prep to forecast:", waste_report(test.orders, test.predicted, avg_price))
print("model + 16% buffer:", waste_report(test.orders, test.predicted * 1.16, avg_price))

baseline (last week): {'meals_wasted': 1350403, 'kg_wasted': np.float64(540161.2), 'cost_wasted_rs': 138312583, 'meals_unserved': 1246097, 'revenue_lost_rs': 425430764}
model, prep to forecast: {'meals_wasted': 971618, 'kg_wasted': np.float64(388647.1), 'cost_wasted_rs': 99516191, 'meals_unserved': 1145416, 'revenue_lost_rs': 391057111}
model + 16% buffer: {'meals_wasted': 1666758, 'kg_wasted': np.float64(666703.0), 'cost_wasted_rs': 170714624, 'meals_unserved': 674792, 'revenue_lost_rs': 230381159}


My responsibility in WasteWise was building and evaluating the machine-learning model that predicts how many meals customers will order in the following week. I used a Gradient Boosting model, which combines multiple decision trees to learn patterns from historical data and make predictions. The model uses information such as previous orders, meal categories, prices, promotions and kitchen details to forecast future demand. We divided the dataset chronologically into three sections: training (weeks 1–125), validation (weeks 126–135), and testing (weeks 136–145). This ensures that the model learns from past data and is evaluated on future data it hasn't seen before. I then experimented with three different combinations of hyperparameters, changing the number of boosting iterations, learning rate and maximum leaf nodes. I compared their Mean Absolute Error (MAE), which measures how far the predictions are from the actual orders on average, and selected the configuration with the lowest validation MAE of 78.333. Finally, we evaluated our model against two simple baselines: predicting next week's orders using last week's orders and using the average orders from the previous four weeks. Our final reported test MAE was 63.9244, compared with 79.1326 for the last-week baseline and 81.7013 for the four-week average baseline. This means our model made more accurate predictions than both simpler methods. These predictions are then used by the rest of our team to estimate how much food could be wasted or saved, helping kitchens make better preparation decisions and reduce unnecessary food waste.

# Part 4 - Evaluation and Failure Analysis

Let's first see how much our estimated food waste changes based on the amount of food given. Here, I'm trying to test kg weights of 0.3, 0.4, and 0.5 to see how the physical weight of wasted food changes and to convey the impact of wasting food.

In [ ]:

# Using the best model, we generate predictions
my_val_predictions = pd.Series(best_model.predict(X_val), index=val.index)

# Calculate the average meal price in the validation data
my_avg_price = val["price_paid"].mean()

# Run the existing waste report on our model's predictions
my_val_report = waste_report(
    val["orders"],
    my_val_predictions,
    my_avg_price
)

# Get the number of meals estimated to be wasted
my_meals_wasted = my_val_report["meals_wasted"]

# Test different assumptions about meal weight
my_kg_results = []

for kg in [0.3, 0.4, 0.5]:

    # Calculate estimated kilograms of food wasted
    estimated_kg = my_meals_wasted * kg

    # Store the results
    my_kg_results.append({
        "KG_PER_MEAL": kg,
        "Meals Wasted": my_meals_wasted,
        "Estimated KG Wasted": round(estimated_kg, 1),
        "Food Cost (Rs)": my_val_report["cost_wasted_rs"],
        "Revenue Lost (Rs)": my_val_report["revenue_lost_rs"]
    })

# Convert our results into a readable table
my_kg_table = pd.DataFrame(my_kg_results)

# Display the table
print("KG_PER_MEAL SENSITIVITY ANALYSIS")
display(my_kg_table)

# Convert the waste report dictionary into a DataFrame
my_waste_summary = pd.DataFrame([my_val_report])

# Display the complete waste report
print("FINAL MODEL — VALIDATION WASTE REPORT")
display(my_waste_summary)


KG_PER_MEAL SENSITIVITY ANALYSIS


,KG_PER_MEAL,Meals Wasted,Estimated KG Wasted,Food Cost (Rs),Revenue Lost (Rs)
0,0.3,1199758,359927.4,122048449,469285500
1,0.4,1199758,479903.2,122048449,469285500
2,0.5,1199758,599879.0,122048449,469285500


FINAL MODEL — VALIDATION WASTE REPORT


,meals_wasted,kg_wasted,cost_wasted_rs,meals_unserved,revenue_lost_rs
0,1199758,479903.1,122048449,1383948,469285500


We have the same meals wasted, food cost, and revenue cost, due to the fact that waste_report() has no area of which it depends on KG_PER_MEAL. This means that regardless of the weight of food, the losses will be the same.

In [ ]:
# Generate predictions for the test dataset using our best model
test_predictions_series = pd.Series(test_predictions, index=test.index)

# Calculate average price in the test set
test_avg_price = test["price_paid"].mean()

# Run the waste report comparing actual orders vs. model predictions on test data
model_test_waste = waste_report(
    test["orders"],
    test_predictions_series,
    test_avg_price
)

baseline_1_report = waste_report(
    test["orders"],
    test["orders_last_week"],
    test_avg_price
)

baseline_2_report = waste_report(
    test["orders"],
    test["orders_avg_4wk"],
    test_avg_price
)

comparison_table = pd.DataFrame({"ML model":model_test_waste, "Last week":baseline_1_report, "Four weeks ago":baseline_2_report})

display(comparison_table)


,ML model,Last week,Four weeks ago
meals_wasted,971618.0,1350403.0,1500708.0
kg_wasted,388647.1,540161.2,600283.0
cost_wasted_rs,99516191.0,138312583.0,153707249.0
meals_unserved,1145416.0,1246097.0,1180074.0
revenue_lost_rs,391057111.0,425430764.0,402889981.0


Preparing the predicted amount of meals by the models may not be the smartest financial decision. We are testing different multipliers to see which one lowers the total financial cost even further.

In [ ]:
# Test different buffer multipliers on the validation set predictions
buffer_tuning_results = []

for m in [1.00, 1.05, 1.10, 1.12, 1.14, 1.16, 1.18, 1.20]:
    # Multiply model predictions by the buffer factor
    buffered_preds = my_val_predictions * m

    # Run the waste report for this buffered prediction
    r = waste_report(val["orders"], buffered_preds, my_avg_price)

    # Calculate total financial cost (wasted ingredients + lost revenue)
    total_financial_cost = r["cost_wasted_rs"] + r["revenue_lost_rs"]

    buffer_tuning_results.append({
        "Multiplier": m,
        "Total Cost (Rs)": total_financial_cost,
        "Meals Wasted": r["meals_wasted"],
        "Meals Unserved": r["meals_unserved"]
    })

buffer_tuning_df = pd.DataFrame(buffer_tuning_results)
print("Buffer Multiplier Tuning on Validation Set:")
display(buffer_tuning_df)

# Find the optimal multiplier that resulted in the lowest total cost
optimal_row = buffer_tuning_df.loc[buffer_tuning_df["Total Cost (Rs)"].idxmin()]
print(f"\nOptimal Buffer Multiplier for Model: {optimal_row['Multiplier']}")

Buffer Multiplier Tuning on Validation Set:


,Multiplier,Total Cost (Rs),Meals Wasted,Meals Unserved
0,1.00,591333949,1199758,1383948
1,1.05,551570520,1434810,1196168
2,1.10,522820057,1694845,1033370
3,1.12,514292677,1805603,974995
4,1.14,507253488,1919737,919996
5,1.16,501658553,2037148,868273
6,1.18,497353493,2157484,819477
7,1.20,494500451,2281114,773974



Optimal Buffer Multiplier for Model: 1.2


We are now summarizing all our reports and findings. There is a noticable trend of decreasing financial costs and meals unserved, with and increasing amount of meals wasted. This isn't a coding error, but rather a trade-off since optimizing food costs doesn't necessarily optimize foods wasted.

In [ ]:
# Calculate test metrics for Baselines and Models
test_avg_price = test["price_paid"].mean()

# Baseline 1: Last Week
b1_waste = waste_report(test["orders"], test["orders_last_week"], test_avg_price)
b1_cost = b1_waste["cost_wasted_rs"] + b1_waste["revenue_lost_rs"]

# Baseline 2: 4-Week Average
b2_waste = waste_report(test["orders"], test["orders_avg_4wk"], test_avg_price)
b2_cost = b2_waste["cost_wasted_rs"] + b2_waste["revenue_lost_rs"]

# Model with Exact Forecast
model_exact_waste = waste_report(test["orders"], test_predictions_series, test_avg_price)
model_exact_cost = model_exact_waste["cost_wasted_rs"] + model_exact_waste["revenue_lost_rs"]

# Model + Optimal Buffer
# Let's use the optimal multiplier found
best_multiplier = float(optimal_row["Multiplier"])
model_buffered_preds = test_predictions_series * best_multiplier
model_buffered_waste = waste_report(test["orders"], model_buffered_preds, test_avg_price)
model_buffered_cost = model_buffered_waste["cost_wasted_rs"] + model_buffered_waste["revenue_lost_rs"]

# Compile everything into a final summary table
final_comparison_data = [
    {
        "Strategy": "Baseline (Last Week)",
        "Test MAE": round(test_baseline_1, 2),
        "Kg Wasted": round(b1_waste["kg_wasted"]),
        "Total Cost (Rs)": b1_cost
    },
    {
        "Strategy": "Baseline (4-Week Avg)",
        "Test MAE": round(test_baseline_2, 2),
        "Kg Wasted": round(b2_waste["kg_wasted"]),
        "Total Cost (Rs)": b2_cost
    },
    {
        "Strategy": "ML Model (Exact Forecast)",
        "Test MAE": round(test_model_mae, 2),
        "Kg Wasted": round(model_exact_waste["kg_wasted"]),
        "Total Cost (Rs)": model_exact_cost
    },
    {
        "Strategy": f"ML Model + Buffer ({best_multiplier})",
        "Test MAE": round(test_model_mae, 2), # MAE stays same, prepping is scaled
        "Kg Wasted": round(model_buffered_waste["kg_wasted"]),
        "Total Cost (Rs)": model_buffered_cost
    }
]

final_comparison_table = pd.DataFrame(final_comparison_data)
print("FINAL PERFORMANCE & WASTE COMPARISON TABLE")
display(final_comparison_table)

FINAL PERFORMANCE & WASTE COMPARISON TABLE


,Strategy,Test MAE,Kg Wasted,Total Cost (Rs)
0,Baseline (Last Week),79.13,540161,563743347
1,Baseline (4-Week Avg),81.70,600283,556597230
2,ML Model (Exact Forecast),64.52,388647,490573302
3,ML Model + Buffer (1.2),64.52,749378,393329225


Let's see where the model makes mistakes, such as how accurate the model is, whether the prediction is too high or too low, and areas of biggest mistakes.

In [ ]:
# Create a DataFrame to analyze prediction errors
error_analysis = pd.DataFrame({
    "Actual Orders": test["orders"],
    "Predicted Orders": test_predictions_series,
    "Category": test["category"],
    "Cuisine": test["cuisine"]
})

# Calculate prediction errors
error_analysis["Error"] = (
    error_analysis["Predicted Orders"] -
    error_analysis["Actual Orders"]
)

# Calculate absolute prediction errors
error_analysis["Absolute Error"] = error_analysis["Error"].abs()

# Classify predictions
error_analysis["Prediction Type"] = error_analysis["Error"].apply(
    lambda x: "Overprediction" if x > 0
    else ("Underprediction" if x < 0 else "Exact Prediction")
)

# Display summary statistics
print("OVERALL PREDICTION ERROR ANALYSIS")

print("Mean Absolute Error:",
      error_analysis["Absolute Error"].mean())

print("Mean Prediction Error:",
      error_analysis["Error"].mean())

print("\nPrediction Types:")
display(error_analysis["Prediction Type"].value_counts())


# 5. Average error magnitude by prediction type
print("\nAVERAGE ERROR BY PREDICTION TYPE")

print("Average Overprediction:",
      error_analysis.loc[
          error_analysis["Error"] > 0, "Error"
      ].mean())

print("Average Underprediction:",
      error_analysis.loc[
          error_analysis["Error"] < 0, "Error"
      ].mean())

# 6. Prediction errors by meal category
print("\nPREDICTION ERRORS BY MEAL CATEGORY")

category_errors = error_analysis.groupby("Category").agg(
    MAE=("Absolute Error", "mean"),
    Mean_Error=("Error", "mean"),
    Total_Predictions=("Error", "count")
).sort_values("MAE", ascending=False)

display(category_errors)

# 7. Prediction errors by cuisine
print("\nPREDICTION ERRORS BY CUISINE")

cuisine_errors = error_analysis.groupby("Cuisine").agg(
    MAE=("Absolute Error", "mean"),
    Mean_Error=("Error", "mean"),
    Total_Predictions=("Error", "count")
).sort_values("MAE", ascending=False)

display(cuisine_errors)

# 8. Identify the largest prediction errors
print("\nTOP 10 LARGEST PREDICTION ERRORS")

largest_errors = error_analysis.reindex(
    error_analysis["Absolute Error"].nlargest(10).index
)

display(largest_errors)


OVERALL PREDICTION ERROR ANALYSIS
Mean Absolute Error: 64.52009950203652
Mean Prediction Error: -5.296780734429124

Prediction Types:


,count
Prediction Type,
Overprediction,17724
Underprediction,15088



AVERAGE ERROR BY PREDICTION TYPE
Average Overprediction: 54.81932790009969
Average Underprediction: -75.91567717122582

PREDICTION ERRORS BY MEAL CATEGORY


,MAE,Mean_Error,Total_Predictions
Category,,,
Rice Bowl,135.058689,-4.569999,2310
Sandwich,108.756514,-45.950063,2306
Salad,94.408646,31.524602,2310
Pizza,87.431079,-41.210980,2306
Extras,67.801926,4.895018,994
Beverages,66.909979,-9.093682,8981
Other Snacks,51.557835,5.850059,2095
Fish,39.664565,-8.340194,1367
Soup,34.436591,8.725570,918



PREDICTION ERRORS BY CUISINE


,MAE,Mean_Error,Total_Predictions
Cuisine,,,
Italian,82.293267,-8.180692,8853
Indian,59.540535,2.151020,8020
Thai,58.811956,-1.060602,8310
Continental,55.347853,-14.394008,7629



TOP 10 LARGEST PREDICTION ERRORS


,Actual Orders,Predicted Orders,Category,Cuisine,Error,Absolute Error,Prediction Type
428107,67,2124.737955,Sandwich,Italian,2057.737955,2057.737955,Overprediction
437058,3631,2129.260430,Beverages,Thai,-1501.739570,1501.739570,Underprediction
440769,2174,804.731309,Pizza,Continental,-1369.268691,1369.268691,Underprediction
452810,2715,1407.148903,Rice Bowl,Indian,-1307.851097,1307.851097,Underprediction
450193,636,1920.836325,Salad,Italian,1284.836325,1284.836325,Overprediction
453496,703,1962.509858,Beverages,Thai,1259.509858,1259.509858,Overprediction
448674,1376,157.007232,Beverages,Italian,-1218.992768,1218.992768,Underprediction
442993,2067,952.913266,Pizza,Continental,-1114.086734,1114.086734,Underprediction
440289,1647,558.655855,Pizza,Continental,-1088.344145,1088.344145,Underprediction
439697,2485,1398.994388,Rice Bowl,Indian,-1086.005612,1086.005612,Underprediction


This can shows that while our Gradient Model did greatly improve our baseline stats, it still has limitations across different meals.

Our mean prediction error is -5.30, so its underpredictions are larger on average, even though the overpredictions happened more often.

The 1.20 buffer decreased the food costs by 19.8%, but increased the food wasted by around 92.8%, favoring the finance aspect of the structure.

There are some meals with very high MAEs, such as the rice bowl and sandwiches, while the biriyani has a low MAE, so the accuracy really depends on the meal being predicted.


# Part 5 - User Interface

In [ ]:
d = test.copy()
d["m"] = (best_model.predict(d[FEATURES]) - d.orders).clip(lower=0)
d["b"] = (d.orders_last_week - d.orders).clip(lower=0)
d["saved"] = d.b - d.m
print(d.nlargest(5, "saved")[["kitchen","meal","week","orders","orders_last_week","saved"]])

        kitchen  meal  week  orders  orders_last_week        saved
425140      157  2290   136     541            2983.0  1880.183206
447268       89  1962   143     296            1729.0  1433.000000
443004       43  2490   141     609            2011.0  1359.056243
432951       10  1727   138     891            2188.0  1254.699867
433974      186  2290   139     338            1904.0  1203.528435


In [ ]:
!pip install gradio -q
import gradio as gr

Looking up the row. len(row) == 0 means that certain combination does not exist, only if all of them mataches then

In [ ]:
def get_row(kitchen, meal, week):
    row = test[(test.kitchen == kitchen) &
               (test.meal == meal) &
               (test.week == week)]
    return row if len(row) > 0 else None

Recommendation - the function Gradio calls

In [ ]:
def recommend(kitchen, meal, week):
    row = get_row(int(kitchen), int(meal), int(week))
    if row is None:
        return "No data for that kitchen, meal and week."

    forecast  = best_model.predict(row[FEATURES])[0] # giving features to already trained model to predict demand
    prep      = forecast * OPTIMAL_MULTIPLIER #
    baseline  = row.orders_last_week.values[0]
    actual    = row.orders.values[0]

    model_over    = max(0, forecast - actual)
    baseline_over = max(0, baseline - actual)
    kg_saved      = (baseline_over - model_over) * KG_PER_MEAL

    return (
        f"WEEK {int(week)} RECOMMENDATION\n\n"
        f"Predicted orders: {forecast:.0f}\n"
        f"Prepare: {prep:.0f} meals  (forecast + {int((OPTIMAL_MULTIPLIER-1)*100)}% safety buffer)\n\n"
        f"— How this compares —\n"
        f"Without our model, prepping to last week's number: {baseline:.0f}\n"
        f"What customers actually ordered: {actual:.0f}\n\n"
        f"Food waste avoided: {kg_saved:.1f} kg"
    )

Interface

In [ ]:
gr.Interface(
    fn=recommend,
    inputs=[gr.Number(label="Kitchen ID", value=157),
        gr.Number(label="Meal ID", value=2290),
        gr.Number(label="Week (136-145)", value=136)],
    outputs=gr.Textbox(label="Recommendation", lines=8),
    title="WasteWise",
    description="Weekly prep recommendations that cut food waste."
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fff78c9b8f19b55c4b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
check = df[(df.kitchen == 157) & (df.meal == 2290) & (df.week >= 130)]
check[["week", "orders", "price_cut", "promo_email", "on_homepage"]]

,week,orders,price_cut,promo_email,on_homepage
405402,130,163,0.000000,0,0
408670,131,351,0.000000,0,0
411943,132,351,0.162607,0,0
415252,133,796,0.148633,0,0
418576,134,5577,0.244068,1,0
421879,135,2983,0.333264,0,1
425140,136,541,0.272879,0,0
428431,137,513,0.213559,0,0
431698,138,539,0.189169,0,0
434978,139,445,0.158872,0,0


In [ ]:
buffered_mae = mae(test["orders"], test_predictions_series * 1.2)
print("MAE, model + 20% buffer:", round(buffered_mae, 2))

MAE, model + 20% buffer: 75.08


### Small Part ( Carbon Emissons )

In [ ]:
# Carbon impact of avoided food waste
# Emission factor: 3.07 kg CO2e per kg of food (FLW Protocol weighted average)

kg_saved = 540161 - 388647
print("Food waste avoided:", kg_saved, "kg")

print("Low estimate (factor 2.0):", kg_saved * 2.0 / 1000, "tonnes CO2e")
print("Main estimate (factor 3.07):", kg_saved * 3.07 / 1000, "tonnes CO2e")
print("High estimate (factor 5.0):", kg_saved * 5.0 / 1000, "tonnes CO2e")